# PolyWhisper Evaluation
Run after training. Computes WER on FLEURS test set.

In [ ]:
# CELL 1: Install + Imports
!pip install -q transformers==4.44.2 datasets==3.1.0 soundfile librosa jiwer

import torch, torch.nn as nn, numpy as np, json, math, re
from pathlib import Path
from tqdm import tqdm
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from datasets import load_dataset
from google.colab import drive
from jiwer import wer

print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")

In [ ]:
# CELL 2: Config + Model (same as training)

WHISPER_MODEL = "openai/whisper-base"
LANGUAGES = ["en", "hi"]
MAX_AUDIO_SEC = 15.0
MAX_LABEL_LEN = 128
AD_HID = 256
AD_LAYERS = 2
AD_HEADS = 4
AD_FFN = 1024
AD_RANK = 8

drive.mount("/content/drive")
SAVE_DIR = Path("/content/drive/MyDrive/polywhisper")
ADAPTER_DIR = SAVE_DIR / "adapters"
DATA_DIR = SAVE_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

processor = WhisperProcessor.from_pretrained(WHISPER_MODEL)
ENCODE_DIM = 512
VOCAB_SIZE = len(processor.tokenizer)
tok = processor.tokenizer

class LowRank(nn.Module):
    def __init__(self, i, o, r):
        super().__init__()
        self.a = nn.Linear(i, r, bias=False)
        self.b = nn.Linear(r, o, bias=False)
        nn.init.zeros_(self.b.weight)
    def forward(self, x):
        return self.b(self.a(x))

class SelfAttn(nn.Module):
    def __init__(self, d, h, r):
        super().__init__()
        self.q = nn.Linear(d, d)
        self.k = nn.Linear(d, d)
        self.v = nn.Linear(d, d)
        self.o = nn.Linear(d, d)
        self.qa = LowRank(d, d, r)
        self.va = LowRank(d, d, r)
        self.h, self.dh = h, d // h
        self.sc = math.sqrt(self.dh)
        self.drop = nn.Dropout(0.1)
    def forward(self, x):
        B, T, _ = x.shape
        q = self.q(x) + self.qa(x)
        k, v = self.k(x), self.v(x) + self.va(x)
        def rs(t): return t.view(B, T, self.h, self.dh).transpose(1, 2)
        q, k, v = rs(q), rs(k), rs(v)
        a = self.drop(torch.softmax((q @ k.transpose(-2, -1)) / self.sc, dim=-1))
        return self.o((a @ v).transpose(1, 2).contiguous().view(B, T, -1))

class CrossAttn(nn.Module):
    def __init__(self, d, ed, h, r):
        super().__init__()
        self.q = nn.Linear(d, d)
        self.k = nn.Linear(ed, d)
        self.v = nn.Linear(ed, d)
        self.o = nn.Linear(d, d)
        self.qa = LowRank(d, d, r)
        self.va = LowRank(ed, d, r)
        self.h, self.dh = h, d // h
        self.sc = math.sqrt(self.dh)
        self.drop = nn.Dropout(0.1)
    def forward(self, x, enc):
        B, Td, _ = x.shape
        Te = enc.size(1)
        q = self.q(x) + self.qa(x)
        k, v = self.k(enc), self.v(enc) + self.va(enc)
        def rs(t, T): return t.view(B, T, self.h, self.dh).transpose(1, 2)
        q, k, v = rs(q, Td), rs(k, Te), rs(v, Te)
        a = self.drop(torch.softmax((q @ k.transpose(-2, -1)) / self.sc, dim=-1))
        return self.o((a @ v).transpose(1, 2).contiguous().view(B, Td, -1))

class AdaptLayer(nn.Module):
    def __init__(self, d, ed, h, ffn, r):
        super().__init__()
        self.sa = SelfAttn(d, h, r)
        self.ca = CrossAttn(d, ed, h, r)
        self.ff = nn.Sequential(nn.Linear(d, ffn), nn.GELU(), nn.Dropout(0.1), nn.Linear(ffn, d))
        self.n1 = nn.LayerNorm(d)
        self.n2 = nn.LayerNorm(d)
        self.n3 = nn.LayerNorm(d)
        self.drop = nn.Dropout(0.1)
    def forward(self, x, enc):
        x = x + self.drop(self.sa(self.n1(x)))
        x = x + self.drop(self.ca(self.n2(x), enc))
        x = x + self.drop(self.ff(self.n3(x)))
        return x

class LangAdapter(nn.Module):
    def __init__(self):
        super().__init__()
        self.te = nn.Embedding(VOCAB_SIZE, AD_HID)
        self.pe = nn.Embedding(MAX_LABEL_LEN, AD_HID)
        self.layers = nn.ModuleList([AdaptLayer(AD_HID, ENCODE_DIM, AD_HEADS, AD_FFN, AD_RANK) for _ in range(AD_LAYERS)])
        self.out = nn.Linear(AD_HID, VOCAB_SIZE)
        self.norm = nn.LayerNorm(AD_HID)
        self.drop = nn.Dropout(0.1)
    def forward(self, enc, ids):
        ids = ids.clamp(min=0, max=VOCAB_SIZE - 1)
        B, T = ids.shape
        pos = torch.arange(T, device=ids.device).unsqueeze(0)
        x = self.drop(self.te(ids) + self.pe(pos))
        for l in self.layers:
            x = l(x, enc)
        return self.out(self.norm(x))

class PolyWhisper(nn.Module):
    def __init__(self):
        super().__init__()
        self.whisper = WhisperForConditionalGeneration.from_pretrained(WHISPER_MODEL)
        for p in self.whisper.model.encoder.parameters():
            p.requires_grad = False
        self.adapters = nn.ModuleDict({l: LangAdapter() for l in LANGUAGES})
    def encode(self, feat):
        return self.whisper.model.encoder(feat).last_hidden_state
    def forward(self, feat, dec_ids, lang):
        enc = self.encode(feat)
        return self.adapters[lang](enc, dec_ids)
    def load_ckpt(self, path, device="cpu"):
        st = torch.load(path, map_location=device, weights_only=True)
        for n, s in st.items():
            if n in self.adapters:
                self.adapters[n].load_state_dict(s)

model = PolyWhisper().to("cuda")
model.eval()

# Load best adapters
for lang in LANGUAGES:
    p = ADAPTER_DIR / f"{lang}_best.pt"
    if p.exists():
        model.load_ckpt(p)
        print(f"Loaded {lang}: {p.name}")
    else:
        print(f"WARNING: No adapter for {lang}")

print(f"Model ready. Params: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.1f}M trainable")

In [ ]:
# CELL 3: Autoregressive Generation

@torch.no_grad()
def generate(model, audio_feat, lang, max_len=128, temperature=1.0):
    enc = model.encode(audio_feat)
    device = audio_feat.device
    # Start with bos token
    dec_ids = torch.tensor([[tok.bos_token_id]], device=device)
    for _ in range(max_len):
        logits = model.adapters[lang](enc, dec_ids)
        next_logits = logits[:, -1, :] / temperature
        next_token = torch.argmax(next_logits, dim=-1, keepdim=True)
        dec_ids = torch.cat([dec_ids, next_token], dim=1)
        if next_token.item() == tok.eos_token_id:
            break
    return tok.decode(dec_ids[0].cpu().tolist(), skip_special_tokens=True)

# Quick test
print("Testing generation...")
dummy = torch.randn(1, 80, 3000).to("cuda")
for lang in LANGUAGES:
    text = generate(model, dummy, lang)
    print(f"  {lang}: \"{text}\"")

print("\nGeneration works!")

In [ ]:
# CELL 4: Load FLEURS Test Data

import soundfile as sf

class FleursTest:
    def __init__(self, lang, split="test"):
        self.cache = DATA_DIR / f"fleurs_{lang}_{split}.json"
        self.adir = DATA_DIR / f"audio_{lang}_{split}"
        self.adir.mkdir(parents=True, exist_ok=True)
        self.data = self._load(lang, split)
    def _load(self, lang, split):
        cfg = {"en": "en_us", "hi": "hi_in"}[lang]
        if self.cache.exists():
            print(f"  Cached {lang}/{split}")
            return json.load(open(self.cache))
        print(f"  Downloading FLEURS {lang}/{split}...")
        ds = load_dataset("google/fleurs", cfg, split=split)
        recs = []
        for i, item in enumerate(tqdm(ds, desc=f"  {lang}/{split}")):
            wav = self.adir / f"{i:05d}.wav"
            a = np.array(item["audio"]["array"], dtype=np.float32)
            a = a[:int(MAX_AUDIO_SEC * 16000)]
            sf.write(str(wav), a, 16000)
            t = item["transcription"]
            recs.append({"wav": str(wav), "text": t.lower().strip() if lang == "en" else t.strip()})
        json.dump(recs, open(self.cache, "w"))
        return recs
    def __len__(self):
        return len(self.data)
    def __getitem__(self, i):
        r = self.data[i]
        a, _ = sf.read(r["wav"])
        return {"audio": a.astype(np.float32), "text": r["text"]}

print("Loading test sets...")
test_sets = {}
for lang in LANGUAGES:
    test_sets[lang] = FleursTest(lang, "test")
    print(f"  {lang}: {len(test_sets[lang])} samples")

In [ ]:
# CELL 5: Evaluate WER

@torch.no_grad()
def evaluate_wer(model, test_set, lang, max_samples=None):
    model.eval()
    references = []
    hypotheses = []
    n = max_samples or len(test_set)
    for i in tqdm(range(n), desc=f"Evaluating {lang}"):
        item = test_set[i]
        audio = item["audio"]
        ref = item["text"]
        # Process audio
        inputs = processor.feature_extractor(
            [audio], sampling_rate=16000, return_tensors="pt",
            padding=True, max_length=int(30 * 16000)
        )
        feat = inputs["input_features"].to("cuda")
        if feat.shape[-1] < 3000:
            feat = torch.nn.functional.pad(feat, (0, 3000 - feat.shape[-1]))
        hyp = generate(model, feat, lang)
        references.append(ref)
        hypotheses.append(hyp)
    error = wer(references, hypotheses)
    return error, references, hypotheses

print("=" * 50)
print("EVALUATION")
print("=" * 50)

results = {}
for lang in LANGUAGES:
    print(f"\n--- {lang.upper()} ---")
    error, refs, hyps = evaluate_wer(model, test_sets[lang], lang)
    results[lang] = {"wer": error, "samples": len(refs)}
    print(f"WER: {error * 100:.1f}%")
    print(f"Samples: {len(refs)}")
    # Show 5 examples
    print("\nSamples:")
    for i in range(min(5, len(refs))):
        print(f"  REF: {refs[i][:80]}")
        print(f"  HYP: {hyps[i][:80]}")
        print()

# Summary
print("=" * 50)
print("SUMMARY")
print("=" * 50)
for lang in LANGUAGES:
    r = results[lang]
    print(f"{lang}: WER = {r['wer']*100:.1f}% ({r['samples']} samples)")

In [ ]:
# CELL 6: Save Results

results_path = SAVE_DIR / "eval_results.json"
json.dump(results, open(results_path, "w"), indent=2)
print(f"Results saved to {results_path}")

# Download
from google.colab import files
files.download(str(results_path))